In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval
except Exception:
    pass

import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
from dataclasses import dataclass

import numpy as np
import polars as pl

from dataeval import Metadata
from dataeval.bias import Balance, Diversity, Parity
from dataeval.data import split_dataset
from dataeval.protocols import DatasetMetadata, DatumMetadata

In [3]:
@dataclass
class BoxTarget:
    """A minimal object detection target: boxes, labels, and scores."""

    boxes: np.ndarray
    labels: np.ndarray
    scores: np.ndarray


WEATHER = ("clear", "rainy", "foggy")
TIME_OF_DAY = ("day", "night")
# Class mix per weather: clear is mostly people, foggy is mostly trucks.
CLASS_MIX = {
    "clear": [0.85, 0.10, 0.05],
    "rainy": [0.10, 0.80, 0.10],
    "foggy": [0.05, 0.15, 0.80],
}


class PatrolDataset:
    """A synthetic detection dataset whose class mix depends on the weather."""

    def __init__(self, images: int) -> None:
        rng = np.random.default_rng(0)
        self._weather = rng.choice(WEATHER, images)
        self._time = rng.choice(TIME_OF_DAY, images)
        # Altitude is correlated with weather, but is better explained by weather than by object class
        self._altitude = np.where(
            self._weather == "foggy",
            rng.uniform(300.0, 400.0, images),
            rng.uniform(50.0, 200.0, images),
        )
        self._counts = rng.integers(2, 5, images)
        # Drawn once in __init__ so every read of an item returns the same values
        self._labels = [
            rng.choice(3, count, p=CLASS_MIX[weather])
            for weather, count in zip(self._weather, self._counts, strict=True)
        ]
        self._areas = [rng.uniform(20.0, 200.0, count) for count in self._counts]
        self.metadata = DatasetMetadata(
            id="patrol-demo",
            index2label={0: "person", 1: "car", 2: "truck"},
        )

    def __len__(self) -> int:
        return len(self._counts)

    def __getitem__(self, index: int) -> tuple[np.ndarray, BoxTarget, DatumMetadata]:
        count = int(self._counts[index])
        target = BoxTarget(
            boxes=np.tile(np.array([[0.0, 0.0, 10.0, 10.0]]), (count, 1)),
            labels=self._labels[index],
            scores=np.ones(count),
        )
        # weather, time_of_day, and altitude_m describe the image; box_area describes each box
        datum_metadata: DatumMetadata = DatumMetadata(
            id=index,
            **{
                "weather": str(self._weather[index]),
                "time_of_day": str(self._time[index]),
                "altitude_m": float(self._altitude[index]),
                "box_area": self._areas[index].tolist(),
            },
        )
        return np.zeros((3, 32, 32), dtype=np.uint8), target, datum_metadata


dataset = PatrolDataset(60)
# Declare explicit bin counts for continuous factors to ensure stable comparisons. No exclusion of
# the per-item `id` is needed: it names the datum rather than describing it, so DataEval keeps it
# out of the factor space on its own (it becomes the reserved `item_id` column, not a factor).
metadata = Metadata(
    dataset,
    continuous_factor_bins={"altitude_m": 3, "box_area": 4},
)

print(f"images:     {metadata.level_counts['unit']}")
print(f"detections: {metadata.level_counts['instance']}")
print(f"factors:    {list(metadata.factor_names)}")
print(f"class axis: {metadata.class_axis} ({metadata.class_axis_source})")

images:     60
detections: 188
factors:    ['altitude_m', 'box_area', 'time_of_day', 'weather']
class axis: class_label (ground_truth)


In [4]:
by_weather = metadata.classed_by("weather")

print(f"axis:     {by_weather.class_axis} ({by_weather.class_axis_source})")
print(f"groups:   {dict(by_weather.index2label)}")
print(f"original: {metadata.class_axis} ({metadata.class_axis_source})")

axis:     weather (derived)
groups:   {0: 'clear', 1: 'foggy', 2: 'rainy'}
original: class_label (ground_truth)


In [5]:
print(f"before: {list(metadata.factor_names)}")
print(f"after:  {list(by_weather.factor_names)}")

before: ['altitude_m', 'box_area', 'time_of_day', 'weather']
after:  ['altitude_m', 'box_area', 'class', 'time_of_day']


In [6]:
result = Balance().evaluate(by_weather)
print(result.balance)

shape: (5, 2)
┌─────────────┬──────────┐
│ factor_name ┆ mi_value │
│ ---         ┆ ---      │
│ cat         ┆ f64      │
╞═════════════╪══════════╡
│ weather     ┆ 1.0      │
│ altitude_m  ┆ 0.578686 │
│ box_area    ┆ 0.0      │
│ class       ┆ 0.430873 │
│ time_of_day ┆ 0.0      │
└─────────────┴──────────┘


In [7]:
print(Balance().evaluate(metadata).balance)

shape: (5, 2)
┌─────────────┬──────────┐
│ factor_name ┆ mi_value │
│ ---         ┆ ---      │
│ cat         ┆ f64      │
╞═════════════╪══════════╡
│ class_label ┆ 1.0      │
│ altitude_m  ┆ 0.246742 │
│ box_area    ┆ 0.00501  │
│ time_of_day ┆ 0.0      │
│ weather     ┆ 0.430864 │
└─────────────┴──────────┘


In [8]:
default = Balance().evaluate(metadata)
print(default.classwise.filter(pl.col("factor_name") == "altitude_m"))

shape: (3, 4)
┌────────────┬─────────────┬──────────┬───────────────┐
│ class_name ┆ factor_name ┆ mi_value ┆ is_imbalanced │
│ ---        ┆ ---         ┆ ---      ┆ ---           │
│ cat        ┆ cat         ┆ f64      ┆ bool          │
╞════════════╪═════════════╪══════════╪═══════════════╡
│ car        ┆ altitude_m  ┆ 0.054098 ┆ false         │
│ person     ┆ altitude_m  ┆ 0.177806 ┆ false         │
│ truck      ┆ altitude_m  ┆ 0.411902 ┆ true          │
└────────────┴─────────────┴──────────┴───────────────┘


In [9]:
print(result.classwise.filter(pl.col("factor_name") == "altitude_m"))

shape: (3, 4)
┌────────────┬─────────────┬──────────┬───────────────┐
│ class_name ┆ factor_name ┆ mi_value ┆ is_imbalanced │
│ ---        ┆ ---         ┆ ---      ┆ ---           │
│ cat        ┆ cat         ┆ f64      ┆ bool          │
╞════════════╪═════════════╪══════════╪═══════════════╡
│ clear      ┆ altitude_m  ┆ 0.23874  ┆ false         │
│ foggy      ┆ altitude_m  ┆ 1.0      ┆ true          │
│ rainy      ┆ altitude_m  ┆ 0.30647  ┆ true          │
└────────────┴─────────────┴──────────┴───────────────┘


In [10]:
try:
    Balance().evaluate(metadata.at("unit"))
except ValueError as error:
    print(f"class axis: {error}")

class axis: class_labels is defined at the 'instance' level, but this metadata is viewed at 'unit', which has no label per row. Use md.at('instance') for the labels, or read them from rows_at('unit')["class_label"], or classed_by(...) to condition on a factor these rows do have.


In [11]:
by_weather_per_image = metadata.at("unit").classed_by("weather")
print(Balance().evaluate(by_weather_per_image).balance)

shape: (3, 2)
┌─────────────┬──────────┐
│ factor_name ┆ mi_value │
│ ---         ┆ ---      │
│ cat         ┆ f64      │
╞═════════════╪══════════╡
│ weather     ┆ 1.0      │
│ altitude_m  ┆ 0.567677 │
│ time_of_day ┆ 0.0      │
└─────────────┴──────────┘


In [12]:
print(
    f"detections view: {by_weather.class_labels.shape[0]} rows, "
    f"fan-out {by_weather.class_axis_info.rows_per_group_entity:.2f}"
)
print(
    f"images view:     {by_weather_per_image.class_labels.shape[0]} rows, "
    f"fan-out {by_weather_per_image.class_axis_info.rows_per_group_entity:.2f}"
)

detections view: 188 rows, fan-out 3.13
images view:     60 rows, fan-out 1.00


In [13]:
print(f"detections view factors: {list(by_weather.factor_names)}")
print(f"images view factors:     {list(by_weather_per_image.factor_names)}")

detections view factors: ['altitude_m', 'box_area', 'class', 'time_of_day']
images view factors:     ['altitude_m', 'time_of_day']


In [14]:
cells = metadata.at("unit").classed_by("weather", "altitude_m")

print(f"axis: {cells.class_axis}")
counts = np.bincount(cells.class_labels, minlength=len(cells.index2label))
for code, name in sorted(cells.index2label.items(), key=lambda item: item[1]):
    print(f"  {name:32s} {counts[code]:3d} images")
print(f"\ncells present: {len(cells.index2label)} of {3 * 3} possible")

axis: weather × altitude_m
  clear × < 168.554                 14 images
  clear × [168.554, 280.648)         5 images
  foggy × >= 280.648                20 images
  rainy × < 168.554                 14 images
  rainy × [168.554, 280.648)         7 images

cells present: 5 of 9 possible


In [15]:
parity = Parity().evaluate(cells)
print(parity.factors)
print(f"\nthin cells: {parity.insufficient_data}")

shape: (1, 5)
┌─────────────┬───────┬──────────┬────────────────┬───────────────────────┐
│ factor_name ┆ score ┆ p_value  ┆ is_significant ┆ has_insufficient_data │
│ ---         ┆ ---   ┆ ---      ┆ ---            ┆ ---                   │
│ cat         ┆ f64   ┆ f64      ┆ bool           ┆ bool                  │
╞═════════════╪═══════╪══════════╪════════════════╪═══════════════════════╡
│ time_of_day ┆ 0.0   ┆ 0.619474 ┆ false          ┆ true                  │
└─────────────┴───────┴──────────┴────────────────┴───────────────────────┘

thin cells: {'time_of_day': {'day': {'clear × [168.554, 280.648)': 3}, 'night': {'clear × [168.554, 280.648)': 2, 'rainy × [168.554, 280.648)': 2}}}


/tmp/ipykernel_5458/3923487337.py:1: ExperimentalWarning: 'Parity' is experimental and may change or be removed in any future release without notice.
  parity = Parity().evaluate(cells)
/builds/jatic/aria/dataeval/src/dataeval/bias/_parity.py:255: ExperimentalWarning: 'parity' is experimental and may change or be removed in any future release without notice.
  output = parity(factor_data, class_labels)
/builds/jatic/aria/dataeval/src/dataeval/bias/_parity.py:312: ExperimentalWarning: 'ParityOutput' is experimental and may change or be removed in any future release without notice.
  return ParityOutput(factors=factors_df, insufficient_data=insufficient_data, class_axis=record)


In [16]:
by_conditions = metadata.classed_by("weather", "time_of_day")
print(Diversity().evaluate(by_conditions).classwise.head(6))

shape: (6, 4)
┌───────────────┬─────────────┬─────────────────┬──────────────────┐
│ class_name    ┆ factor_name ┆ diversity_value ┆ is_low_diversity │
│ ---           ┆ ---         ┆ ---             ┆ ---              │
│ cat           ┆ cat         ┆ f64             ┆ bool             │
╞═══════════════╪═════════════╪═════════════════╪══════════════════╡
│ clear × day   ┆ altitude_m  ┆ 0.371134        ┆ true             │
│ clear × day   ┆ box_area    ┆ 0.828179        ┆ false            │
│ clear × day   ┆ class       ┆ 0.128253        ┆ true             │
│ clear × night ┆ altitude_m  ┆ 0.199667        ┆ true             │
│ clear × night ┆ box_area    ┆ 0.923767        ┆ false            │
│ clear × night ┆ class       ┆ 0.213922        ┆ true             │
└───────────────┴─────────────┴─────────────────┴──────────────────┘


In [17]:
axis = result.class_axis
assert axis
print(f"name:        {axis.name}")
print(f"source:      {axis.source}")
print(f"level:       {axis.level}")
print(f"groups:      {axis.groups}")
print(f"fan-out:     {axis.rows_per_group_entity:.2f} rows per {axis.level}")
print(f"vocabulary:  {axis.vocabulary}")

name:        weather
source:      derived
level:       unit
groups:      3
fan-out:     3.13 rows per unit
vocabulary:  observed


In [18]:
print({k: v for k, v in result.meta().state.items() if k.startswith("class_axis")})

{'class_axis': 'weather', 'class_axis_source': 'derived', 'class_axis_level': 'unit'}


In [19]:
splits = split_dataset(by_weather_per_image, num_folds=2, stratify=True)
assert splits.class_axis
print(f"stratified on: {splits.class_axis.name} ({splits.class_axis.source})")

stratified on: weather (derived)
